In [ ]:
import utils.data as data

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from capstone import *
import pefile

from os import PathLike

dataset = data.MalwareDataset()

mpl.rcParams["figure.dpi"] = 300

In [ ]:
def get_bitness_and_rip(data):
    try:
        pe = pefile.PE(data=data)

        if pe.FILE_HEADER.Machine == pefile.MACHINE_TYPE["IMAGE_FILE_MACHINE_I386"]:
            bitness = 32
        elif pe.FILE_HEADER.Machine == pefile.MACHINE_TYPE["IMAGE_FILE_MACHINE_AMD64"]:
            bitness = 64
        else:
            bitness = "Unknown"

        entry_point = pe.OPTIONAL_HEADER.AddressOfEntryPoint

        pe.close()

        return bitness, entry_point

    except pefile.PEFormatError as e:
        print(f"Error: {e}")
        return None, None

In [ ]:
from iced_x86 import *

# Format specifiers example:
# xchg [rdx+rsi+16h],ah
# xchg %ah,0x16(%rdx,%rsi)
# xchg [rdx+rsi+16h],ah
# xchg ah,[rdx+rsi+16h]
# xchg ah,[rdx+rsi+16h]
# xchgb %ah, %ds:0x16(%rdx,%rsi)


def dissassemble(path: PathLike):
    data = open(path, "rb").read()
    bitness, rip = get_bitness_and_rip(data)

    decoder = Decoder(bitness, data, ip=rip)

    formatter = Formatter(FormatterSyntax.NASM)

    for instr in decoder:
        print(instr.code)

        # disasm = formatter.format(instr)
        # print(formatter.format_mnemonic(instr))

    # ====== =============================================================================
    # F-Spec Description
    # ====== =============================================================================
    # f      Fast formatter (masm-like syntax)
    # g      GNU Assembler formatter
    # i      Intel (XED) formatter
    # m      masm formatter
    # n      nasm formatter
    # X      Uppercase hex numbers with ``0x`` prefix
    # x      Lowercase hex numbers with ``0x`` prefix
    # H      Uppercase hex numbers with ``h`` suffix
    # h      Lowercase hex numbers with ``h`` suffix
    # r      RIP-relative memory operands use RIP register instead of abs addr (``[rip+123h]`` vs ``[123456789ABCDEF0h]``)
    # U      Uppercase everything except numbers and hex prefixes/suffixes (ignored by fast fmt)
    # s      Add a space after the operand separator
    # S      Always show the segment register (memory operands)
    # B      Don't show the branch size (``SHORT`` or ``NEAR PTR``) (ignored by fast fmt)
    # G      (GNU Assembler): Add mnemonic size suffix (eg. ``movl`` vs ``mov``)
    # M      Always show the memory size (eg. ``BYTE PTR``) even when not needed
    # _      Use digit separators (eg. ``0x12345678`` vs ``0x1234_5678``) (ignored by fast fmt)
    # ====== =============================================================================


dissassemble(dataset[0][0])